# Detailed v3 Re-Benchmark Audit (tabular + generated cases)

Re-runs all live v3 benchmarks at maximum detail and writes one organised,
Drive-accessible audit folder. Two parts, in order:

1. Tabular - every relevant metric per agent (triage acuity+disposition,
 doctor diagnosis/department base+nurse, doctor disposition, Stage-2 exact-ICD),
 into `tabular_full.json` + per-agent CSVs, with a `regression_check.json` that
 compares each headline metric to `ceiling.json` (expected all-PASS).
2. Generated cases (`--parser-llm --skip-e2e`) for both backends
 (Flash + MedGemma): rich per-backend JSON + per-agent CSVs (incl. graded ICD:
 TF-IDF / Gemini / ICD-tree) + `per_case_master.csv`, then a Flash-vs-MedGemma
 head-to-head.

All outputs land under (on your Drive, via the `artifacts/` symlink):
```
artifacts/benchmarks/audit/<YYYY-MM-DD>/
 tabular/ tabular_full.json, regression_check.json, *.csv, icd_resolution_full.json
 generated_cases/
 flash/ e2e_full_flash.json + per-agent CSVs
 medgemma/ e2e_full_medgemma.json + per-agent CSVs
 comparison/ backend_comparison.json + *.csv
```


## Required Drive structure (same as the training notebooks)

```
MyDrive/proiect_licenta/
 data/ mimic-iv-ed/ (triage,edstays,vitalsign,medrecon, files_created/categorized_diagnosis)
 mimic-iv/hosp/ (patients,services,diagnoses_icd,admissions)
 mimic-iv-notes/.../discharge.csv (~3.3 GB)
 derived/synthetic_cases/cases.json (+ sampled_features.pkl) <- the 250 generated cases
 artifacts/ triage/v3/ doctor/v3_base/ doctor/v3/ (+ disposition_model*.joblib, icd_resolver/)
```

## Notes / known gotchas
- Inference only - no GPU needed. A CPU runtime is fine (and avoids GPU quota).
- Runtime is long on the FIRST run only. The tabular section streams `discharge.csv`
 (3.3 GB) for the PMH parse. A loader disk-cache on Drive (set up in Cell 7 via
 `LOADER_CACHE_DIR`) saves each cleaned frame on first build, so re-runs and later
 Colab sessions load them in seconds. Budget ~1.5-2.5 h the first time; minutes after.
 The cache auto-invalidates if a MIMIC source file changes.
- `GEMINI_API_KEY` is only needed if the ICD title-embedding cache is missing
 entries (the graded "gemini" engine); otherwise it makes zero API calls. Set it in
 the optional cell below to be safe.
- MedGemma needs the vLLM tunnel from `colab/serve_medgemma_colab.ipynb` running;
 paste its `MEDGEMMA_BASE_URL` into the MedGemma cell.
- MRN history lookup (the `tool_direct_lookup` column) needs the history index built
 on the 250 cases - already done per your last rebuild.


## Setup & checks
Run every cell once per Colab session.

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Clone / update the repo
GITHUB_USERNAME = '<YOUR_USERNAME>'  # <- EDIT THIS
REPO_URL     = f'https://github.com/{GITHUB_USERNAME}/licenta.git'
CLONE_PATH   = '/content/licenta'
PROJECT_PATH = f'{CLONE_PATH}/proiect_licenta'
BRANCH       = 'main'

import os, subprocess
if os.path.exists(CLONE_PATH):
    subprocess.run(['git', '-C', CLONE_PATH, 'fetch'], check=False)
    subprocess.run(['git', '-C', CLONE_PATH, 'checkout', BRANCH], check=False)
    subprocess.run(['git', '-C', CLONE_PATH, 'pull'], check=False)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, CLONE_PATH], check=True)
assert os.path.exists(PROJECT_PATH), f'expected nested project at {PROJECT_PATH}'
subprocess.run(['git', '-C', CLONE_PATH, 'log', '-1', '--oneline'])
print('Project root:', PROJECT_PATH)

In [ ]:
# Cell 3: Symlink data/ and artifacts/ from Drive into the project
DRIVE_PROJECT = '/content/drive/MyDrive/proiect_licenta'  # <- EDIT IF NEEDED
for folder in ('data', 'artifacts'):
    src, dest = f'{DRIVE_PROJECT}/{folder}', f'{PROJECT_PATH}/{folder}'
    if os.path.islink(dest):
        print('symlink exists:', dest, '->', os.readlink(dest))
    elif os.path.exists(dest):
        print('WARNING: real dir, not symlink:', dest)
    elif not os.path.exists(src):
        print('WARNING: missing on Drive:', src)
    else:
        os.symlink(src, dest); print('linked', dest, '->', src)
print('Symlinks ready.')

In [ ]:
# Cell 4: Install the package + runtime deps (inference only)
import sys
def pip(*a): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], check=True)
pip('-e', PROJECT_PATH)
pip('xgboost>=2.0.0')
pip('thefuzz[speedup]>=0.22.0')
pip('scikit-learn>=1.6.0')
pip('tqdm>=4.66.0', 'python-dotenv>=1.0.0')
pip('google-genai>=0.3.0')          # only used if the Gemini title-embedding cache misses
print('Dependencies installed.')

In [ ]:
# Cell 5 (OPTIONAL): Gemini key - only needed if the ICD title-embedding cache
# is missing titles (graded 'gemini' engine). Skip if the cache is complete.
import os
# from google.colab import userdata
# os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('GEMINI_API_KEY set:', bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')))

In [ ]:
# Cell 6: Verify required files, artifacts, the 250 cases, caches, and the index
sys.path.insert(0, f'{PROJECT_PATH}/src')
from proiect_licenta.paths import (
    TRIAGE_CSV, EDSTAYS_CSV, PATIENTS_CSV, DIAGNOSIS_CSV, SERVICES_CSV,
    VITALSIGN_CSV, MEDRECON_CSV, DIAGNOSES_ICD_CSV, ADMISSIONS_CSV,
    DISCHARGE_NOTES_CSV, TRIAGE_V3_DIR, DOCTOR_V3_BASE_DIR, DOCTOR_V3_DIR,
    DOCTOR_V3_ICD_RESOLVER_DIR, ARTIFACTS_DIR,
)
from pathlib import Path
checks = {
    'triage.csv': TRIAGE_CSV, 'edstays.csv': EDSTAYS_CSV, 'patients.csv': PATIENTS_CSV,
    'categorized_diagnosis.csv': DIAGNOSIS_CSV, 'services.csv': SERVICES_CSV,
    'vitalsign.csv': VITALSIGN_CSV, 'medrecon.csv': MEDRECON_CSV,
    'diagnoses_icd.csv': DIAGNOSES_ICD_CSV, 'admissions.csv': ADMISSIONS_CSV,
    'discharge.csv (3.3GB)': DISCHARGE_NOTES_CSV,
    'triage/v3/': TRIAGE_V3_DIR, 'doctor/v3_base/': DOCTOR_V3_BASE_DIR,
    'doctor/v3/': DOCTOR_V3_DIR,
    'doctor/v3/disposition_model.joblib': DOCTOR_V3_DIR / 'disposition_model.joblib',
    'icd_resolver/': DOCTOR_V3_ICD_RESOLVER_DIR,
    'icd title_embeddings.joblib': DOCTOR_V3_ICD_RESOLVER_DIR / 'title_embeddings.joblib',
}
from proiect_licenta.case_generation import CASES_JSON, FEATURES_PKL
checks['synthetic cases.json'] = CASES_JSON
checks['sampled_features.pkl'] = FEATURES_PKL
missing = []
for name, p in checks.items():
    ok = Path(p).exists()
    print(('OK  ' if ok else 'MISS'), name, '->', p)
    if not ok: missing.append(name)
# Count the generated cases
import json as _json
if CASES_JSON.exists():
    n = len(_json.loads(Path(CASES_JSON).read_text())['cases'])
    print(f'\nGenerated cases: {n} (expected ~250)')
if missing:
    print('\nWARNING missing:', missing)
else:
    print('\nAll required inputs present.')

In [ ]:
# Cell 7: Output folder (dated) + the loader disk-cache, both on Drive
from datetime import date
from proiect_licenta.paths import DERIVED_DIR
AUDIT_BASE = ARTIFACTS_DIR / 'benchmarks' / 'audit' / date.today().isoformat()
TAB_OUT      = AUDIT_BASE / 'tabular'
GEN_BASE     = AUDIT_BASE / 'generated_cases'
FLASH_OUT    = GEN_BASE / 'flash'
MEDGEMMA_OUT = GEN_BASE / 'medgemma'
CMP_OUT      = GEN_BASE / 'comparison'
for d in (TAB_OUT, FLASH_OUT, MEDGEMMA_OUT, CMP_OUT):
    d.mkdir(parents=True, exist_ok=True)

# Loader cache: the heavy load_and_clean_data() loaders re-stream discharge.csv
# (3.3 GB) for the PMH parse. With LOADER_CACHE_DIR set (to this Drive folder),
# the FIRST run builds + saves the cleaned frames there; EVERY later run (and
# later Colab session) loads them in seconds. Auto-invalidates if a MIMIC file
# changes. Unset it to force a fresh parse.
LOADER_CACHE = DERIVED_DIR / 'loader_cache'
LOADER_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['LOADER_CACHE_DIR'] = str(LOADER_CACHE)
print('Audit base  :', AUDIT_BASE)
print('Loader cache:', LOADER_CACHE, '(set as LOADER_CACHE_DIR)')

## Tabular benchmark (detailed)

Reproduces every headline number in `ceiling.json` and expands to the full metric
suite. Long-running (discharge.csv PMH parse per agent). The final
`regression_check.json` must be all-PASS.

> Tip: you can run a subset with `--skip-triage / --skip-doctor / --skip-disposition / --skip-icd`.

In [ ]:
# Cell 8: Run the detailed tabular benchmark (streams live output).
# LOADER_CACHE_DIR is passed explicitly so the subprocess builds/reuses the
# Drive loader cache (first run parses discharge.csv; re-runs load in seconds).
!cd {PROJECT_PATH} && LOADER_CACHE_DIR="{LOADER_CACHE}" python benchmarks/benchmark_tabular_full.py --out-dir "{TAB_OUT}" 

In [ ]:
# Cell 9: Show the regression check vs ceiling.json
import json
reg = json.loads((TAB_OUT / 'regression_check.json').read_text())
print('STATUS:', reg['status'], '| checked', reg.get('n_checked'), '| failed', reg.get('n_failed'))
import pandas as pd
df = pd.DataFrame(reg['checks'])
pd.set_option('display.max_rows', None, 'display.width', 160)
print(df.to_string(index=False))

## Generated cases: Flash (`--parser-llm --skip-e2e`)

Smoke-run 3 cases first (proves the CSV + graded-ICD plumbing), then the full set.
Parse cache (`parse_cache_flash.json`) keeps LLM calls ~0 on a warm re-run.

In [ ]:
# Cell 10: Flash smoke (3 cases)
!cd {PROJECT_PATH} && python benchmarks/benchmark_pipeline_e2e.py \
    --llm-backend flash --parser-llm --skip-e2e --limit 3 \
    --out-dir "{FLASH_OUT}" 

In [ ]:
# Cell 11: Flash - full 250 cases
!cd {PROJECT_PATH} && python benchmarks/benchmark_pipeline_e2e.py \
    --llm-backend flash --parser-llm --skip-e2e \
    --out-dir "{FLASH_OUT}" 

## Generated cases: MedGemma (`--parser-llm --skip-e2e`)

Prereq: start the MedGemma vLLM tunnel (`colab/serve_medgemma_colab.ipynb`) in a
separate session and paste its URL below as `MEDGEMMA_BASE_URL`
(e.g. `https://xxxx.trycloudflare.com/v1`). Uses `parse_cache_medgemma.json`.

In [ ]:
# Cell 12: Point at the MedGemma tunnel, then smoke-run 3 cases
import os
os.environ['LLM_BACKEND']      = 'medgemma'
os.environ['MEDGEMMA_BASE_URL'] = 'https://<PASTE-TUNNEL>/v1'   # <- EDIT THIS
os.environ['MEDGEMMA_MODEL']    = 'google/medgemma-4b-it'
os.environ['MEDGEMMA_API_KEY']  = 'EMPTY'
assert '<PASTE-TUNNEL>' not in os.environ['MEDGEMMA_BASE_URL'], 'set MEDGEMMA_BASE_URL first'
!cd {PROJECT_PATH} && MEDGEMMA_BASE_URL="$MEDGEMMA_BASE_URL" MEDGEMMA_MODEL="$MEDGEMMA_MODEL" MEDGEMMA_API_KEY="$MEDGEMMA_API_KEY" \
    python benchmarks/benchmark_pipeline_e2e.py \
    --llm-backend medgemma --parser-llm --skip-e2e --limit 3 \
    --out-dir "{MEDGEMMA_OUT}" 

In [ ]:
# Cell 13: MedGemma - full 250 cases
!cd {PROJECT_PATH} && MEDGEMMA_BASE_URL="$MEDGEMMA_BASE_URL" MEDGEMMA_MODEL="$MEDGEMMA_MODEL" MEDGEMMA_API_KEY="$MEDGEMMA_API_KEY" \
    python benchmarks/benchmark_pipeline_e2e.py \
    --llm-backend medgemma --parser-llm --skip-e2e \
    --out-dir "{MEDGEMMA_OUT}" 

## Backend head-to-head (Flash vs MedGemma)

Every comparable metric side-by-side + per-case parser comparison (who parsed each
narrative better).

In [ ]:
# Cell 14: Compare backends
!cd {PROJECT_PATH} && python benchmarks/compare_backends.py \
    --flash-dir "{FLASH_OUT}" --medgemma-dir "{MEDGEMMA_OUT}" --out-dir "{CMP_OUT}" 

## Verify outputs

In [ ]:
# Cell 15: List the audit tree + print a few headlines
import json
for p in sorted(AUDIT_BASE.rglob('*')):
    if p.is_file():
        print(f'{p.stat().st_size/1024:8.1f} KB  {p.relative_to(AUDIT_BASE)}')

print('\n--- TABULAR regression:', json.loads((TAB_OUT/'regression_check.json').read_text())['status'])
cmp = json.loads((CMP_OUT/'backend_comparison.json').read_text())
print(f"--- BACKENDS: {cmp['flash_backend']} vs {cmp['medgemma_backend']} "
      f"({cmp['n_metrics']} metrics, {cmp['n_parser_cases']} parser cases)")
for k, v in list(cmp['headline'].items()):
    print(f'    {k}: flash={v.get(cmp["flash_backend"])}  medgemma={v.get(cmp["medgemma_backend"])}  delta={v.get("delta")}')